# Error Analysis: Hausa Sentiment (Classical Model)

This notebook inspects the behaviour of the best classical model
(word + character TF-IDF + calibrated LinearSVC, test macro-F1 0.782)
on the training split that is present in the clean package.

**Goals**

1. Show representative **successful** and **unsuccessful** predictions.
2. Build a lightweight error taxonomy (short text, emoji-heavy, code-switched, negation-like, neutral confusion).
3. Quantify how often each difficulty group is present among errors vs the whole set.

The same analysis can be repeated on the held-out test set once
`afrisenti_hau_test.csv` is added to `data/processed/`.


## 1. Setup


In [ ]:
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..") if Path("..").joinpath("models").exists() else Path(".")
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"
LABELS = ["positive", "neutral", "negative"]
ID2LAB = {0: "positive", 1: "neutral", 2: "negative"}

train = pd.read_csv(DATA / "afrisenti_hau_train.csv")
clf = joblib.load(MODELS / "classical_best.pkl")
print("Train rows:", len(train))
print("Model:", type(clf))


## 2. Run the classical model on the training split

We use `text_bow` (the same view the model was trained on).
Training-set accuracy is optimistic; the point here is qualitative error patterns, not a new score.


In [ ]:
texts = train["text_bow"].fillna("").astype(str).tolist()
y_true = train["label_id"].values

if hasattr(clf, "predict_proba"):
    proba = clf.predict_proba(texts)
    y_pred = proba.argmax(axis=1)
    conf = proba.max(axis=1)
else:
    y_pred = clf.predict(texts)
    conf = np.ones(len(y_pred))

train = train.copy()
train["pred_id"] = y_pred
train["pred"] = [ID2LAB[i] for i in y_pred]
train["correct"] = train["pred_id"] == train["label_id"]
train["confidence"] = conf

print("Training-set accuracy (optimistic):", train["correct"].mean().round(4))
print("Error rate:", (1 - train["correct"].mean()).round(4))
print()
print("Confusion (rows = true, cols = predicted):")
print(pd.crosstab(train["label"], train["pred"]))


#### Interpretation

Neutral is typically the hardest class (most confusions with both positive and negative).
This matches the original test-set behaviour recorded in the experiment registry.


## 3. Lightweight error taxonomy

We flag examples with simple surface heuristics that were also used in the original project:

| Flag | Rule |
|------|------|
| `very_short` | fewer than 6 tokens |
| `emoji_heavy` | 3 or more emoji |
| `code_switched` | original column `code_switched` is True, or mixed Latin script patterns |
| `negation_like` | contains common Hausa negation cues (`ba … ba`, `babu`, `ba`, `kasa`, …) |
| `neutral_true` | gold label is neutral |


In [ ]:
NEG_RE = re.compile(
    r"\b(ba|babu|ban|basu|baka|baki|bata|bai|basa|kasa|rashin)\b",
    flags=re.IGNORECASE,
)

def token_count(s):
    return len(str(s).split())

train["very_short"] = train["n_tokens"].fillna(0).astype(int) < 6
train["emoji_heavy"] = train["n_emoji"].fillna(0).astype(int) >= 3
train["code_switched_flag"] = train["code_switched"].fillna(False).astype(bool)
train["negation_like"] = train["text_bow"].fillna("").map(lambda t: bool(NEG_RE.search(t)))
train["neutral_true"] = train["label"] == "neutral"

flags = ["very_short", "emoji_heavy", "code_switched_flag", "negation_like", "neutral_true"]
print("Prevalence of flags (full train):")
print(train[flags].mean().round(3))


## 4. Error rate by group


In [ ]:
rows = []
for flag in flags:
    sub = train[train[flag]]
    rows.append({
        "group": flag,
        "n": len(sub),
        "error_rate": (1 - sub["correct"].mean()) if len(sub) else np.nan,
        "mean_confidence_on_errors": sub.loc[~sub["correct"], "confidence"].mean() if (~sub["correct"]).any() else np.nan,
    })
# overall
rows.append({
    "group": "ALL",
    "n": len(train),
    "error_rate": 1 - train["correct"].mean(),
    "mean_confidence_on_errors": train.loc[~train["correct"], "confidence"].mean(),
})
group_table = pd.DataFrame(rows)
group_table


#### Interpretation

Groups whose error rate is clearly higher than the overall rate are the difficulties the model still struggles with.
Neutral gold labels and negation-like wording are expected to be elevated; very short tweets often lack enough signal.


## 5. Representative successful predictions

High-confidence correct predictions from each class.


In [ ]:
def show_examples(df, n=4, cols=None):
    cols = cols or ["label", "pred", "confidence", "text_bow", "n_tokens", "n_emoji"]
    cols = [c for c in cols if c in df.columns]
    return df[cols].head(n)

successes = train[train["correct"]].sort_values("confidence", ascending=False)
print("=== High-confidence SUCCESS: positive ===")
display(show_examples(successes[successes["label"] == "positive"]))
print("=== High-confidence SUCCESS: neutral ===")
display(show_examples(successes[successes["label"] == "neutral"]))
print("=== High-confidence SUCCESS: negative ===")
display(show_examples(successes[successes["label"] == "negative"]))


## 6. Representative unsuccessful predictions

We sample high-confidence errors (the model was sure but wrong) and low-confidence errors.


In [ ]:
errors = train[~train["correct"]].copy()
print("Total errors:", len(errors))

print("\n=== HIGH-confidence errors (model sure, but wrong) ===")
display(show_examples(errors.sort_values("confidence", ascending=False), n=6))

print("\n=== LOW-confidence errors (model uncertain) ===")
display(show_examples(errors.sort_values("confidence", ascending=True), n=6))


#### Interpretation

High-confidence errors are the most serious for deployment: the system can look reliable while being wrong.
Common patterns (from the original project and from samples above) include:

- Neutral gold labelled as positive or negative when the tweet is factual / reporting.
- Negation scope missed (`ba … ba` constructions).
- Code-switched or emoji-only tweets with little lexical signal.
- Short political slogans that look positive on the surface but are sarcastic or critical.


## 7. Neutral-class confusion

Neutral is the class with the lowest per-class F1 on the official test set
(see registry: B4 neutral F1 ≈ 0.746). Here we inspect training-set neutral errors.


In [ ]:
neutral_err = errors[errors["label"] == "neutral"]
print("Neutral errors:", len(neutral_err), "of", (train["label"] == "neutral").sum(), "neutral examples")
print()
print("What the model predicted instead:")
print(neutral_err["pred"].value_counts())
print()
display(show_examples(neutral_err.sort_values("confidence", ascending=False), n=8))


## 8. Export summary tables

Write compact CSVs that the report or a later test-set analysis can re-use.


In [ ]:
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)

group_table.to_csv(out_dir / "error_rate_by_group.csv", index=False)

export_cols = ["text_bow", "label", "pred", "confidence", "correct",
               "very_short", "emoji_heavy", "code_switched_flag", "negation_like", "n_tokens", "n_emoji"]
export_cols = [c for c in export_cols if c in train.columns]
errors[export_cols].to_csv(out_dir / "error_analysis_predictions.csv", index=False)

print("Wrote:")
print(" ", out_dir / "error_rate_by_group.csv")
print(" ", out_dir / "error_analysis_predictions.csv")
print("Error rows exported:", len(errors))


## 9. Summary for the report and demo video

| Finding | Evidence in this notebook |
|---------|---------------------------|
| Neutral is the hardest class | Confusion table + §7 |
| High-confidence errors exist | §6 high-confidence error sample |
| Short / emoji-heavy / negation-like groups are riskier | §4 error-rate-by-group table |
| Classical model is still strong overall | Optimistic train accuracy; official test macro-F1 0.782 in registry |

**For the demo video:** pick 2–3 successes and 2–3 failures from §§5–6, show the Gradio app on the same texts, and explain why the failure cases are hard (neutral reporting tone, negation, code-switching, or missing context).
